# Stage 4: the BioLangFusion appendix comparison

This notebook examines saved BioLangFusion runs that used the paper's **appendix convolution settings**. It asks whether combining DNA, RNA and protein representations improves prediction over a single representation on the same supplied benchmark split. This notebook reads completed fits where their saved records are available, checks those records and compares their saved predictions. It does not train models.

The word *appendix* identifies the model configuration, not an extra step required after the [main Phase A notebook](tier2_blf_phaseA.ipynb). That main notebook uses Equation 1's convolution and an additional RNA baseline averaged to codon resolution. This reader retains the appendix configuration, its original single-encoder comparisons and its separate uncertainty calculation. Changing a setting here would not convert one set of trained models into the other.

A **historical run group**, called a *cohort* in some code and records, means model runs with matching recorded inputs, source code and producing environment. It does not mean a new group of biological samples. The [E. coli execution record](../../docs/run-records.md#2026-10-19t220959z-phasea-ecoli-three-seed-classification-comparison) preserves the completed desktop comparison and explains its differences from the main Phase A protocol.

Choose fungal expression or E. coli classification in Setup. For either benchmark, each row pairs a nucleotide sequence with an expression target. The DNA, RNA and translated-protein inputs are derived from that sequence; they are not three independently measured assays. We first identify the examples and compatible saved runs, then select models using validation scores, compare test performance and examine uncertainty. The [Stage 4 overview](README.md#biolangfusion-phase-a) places this retained comparison alongside the main experiment.


---

## Setup

Prepare the environment using the root README's [notebook dependencies](../../README.md#notebook-dependencies), then select that environment as the kernel. Start inside the repository so the path cell can locate `Code/shared_code/`. Package installation belongs in the setup instructions.

The reader needs the benchmark CSV and existing result JSON files. Cache metadata files are sufficient for its independent input-header checks; it does not open the large embedding arrays. Missing headers still allow a file inventory, but block the final comparison. Training is explicitly disabled, and this notebook does not save or overwrite result files.

### Imports


In [ ]:
# Read result records and construct paths/configuration variants.
import json
import sys
# asdict reads all settings; replace varies one candidate without altering the base.
from dataclasses import asdict, replace
from pathlib import Path

# Plot model scores and paired differences.
import matplotlib.pyplot as plt

# Numerical arrays and comparison tables.
import numpy as np
import pandas as pd

# Render pandas tables in notebook output.
from IPython.display import display


### Project paths and shared helpers

Locate this checkout's `Code` folder, then import the shared dataset and result-reading helpers. The origin checks catch a kernel that has retained modules from a different checkout. The [historical runner](run_blf_phaseA.py) supplies helpers here; importing it does not start its command-line training workflow.


In [ ]:
# Make shared_code importable from stage4/ as well as the repository root.
working_directory = Path.cwd().resolve()
REPO_ROOT = next(
    folder for folder in (working_directory, *working_directory.parents)
    if (folder / "Code" / "shared_code" / "datasets.py").is_file()
)
CODE_DIR = REPO_ROOT / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Reuse result filenames and prediction validation from the historical runner.
from stage4 import run_blf_phaseA as phasea_runner
# Load benchmark rows and describe each encoder input.
from shared_code import datasets
from shared_code.encoders import ENCODERS
# Read saved fits/configuration, modality mapping and fingerprint helpers.
from shared_code.fusion import blf, token_embeddings, train
from shared_code.fusion.token_embeddings import (
    file_fingerprint, input_fingerprint, inputs_for_rows,
)

# A kernel can retain imports from another checkout even after sys.path changes.
for module in (datasets, blf, token_embeddings, train):
    if not Path(module.__file__).resolve().is_relative_to(
        CODE_DIR / "shared_code"
    ):
        raise RuntimeError("Shared modules came from another checkout; restart this notebook's kernel.")
if not Path(phasea_runner.__file__).resolve().is_relative_to(CODE_DIR / "stage4"):
    raise RuntimeError(
        "The Phase A helper came from another checkout; restart this notebook's kernel."
    )


### Choose the saved comparison

`BENCHMARK` selects fungal expression or E. coli classification; the saved default is fungal expression. `SEEDS` identifies the training runs to compare, while `BOOTSTRAP_SEED` and `BOOTSTRAP_DRAWS` control later resampling of their saved test predictions.

The result paths select the retained appendix records, not the main notebook's `results/phaseA/` records. The cache path supplies metadata for the input checks. Changing these reader settings selects evidence to inspect; it does not produce missing fits.


In [ ]:
# Select one retained benchmark; keep its historical candidate settings together.
BENCHMARK = "fungal_expression"  # Or "ecoli_proteins"; review one benchmark at a time.
SEEDS = (0, 1, 2)
POSITIVE_LAMBDAS = (0.01, 0.5, 1.0)
RUN_TRAINING = False
# Resample saved predictions later; these values do not retrain the models.
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0

# This notebook is a reader: changing the flag cannot accidentally launch fitting.
if RUN_TRAINING:
    raise RuntimeError("This notebook only inspects saved results; training remains disabled.")

# Reuse existing files in their recorded locations.
STAGE3_DIR = CODE_DIR / "stage3"
# In a separate worktree, set this to the existing Stage 3 embeddings folder; do not copy caches.
# Missing headers permit inspection, but block the final comparison.
CACHE_ROOT = STAGE3_DIR / "embeddings"
RESULT_DIRECTORY = (
    STAGE3_DIR / "results" / BENCHMARK / "reproducible" if BENCHMARK == "fungal_expression"
    else CODE_DIR / "stage4" / "results" / "blf_phaseA" / BENCHMARK
)


### Match targets, metrics and stored dimensions

The benchmark definitions below state the expected retained split counts, number of model outputs and score units. Fungal regression has one numerical output; E. coli classification has three output classes. These expectations check the saved records before scores from different tasks or configurations can be mixed.


In [ ]:
# Expected task, metric, dimensions, and split counts for the selected benchmark.
BENCHMARK_SETTINGS = {
    "fungal_expression": {
        "task": "regression", "n_outputs": 1, "label": "Fungal expression",
        "splits": {"train": 2153, "val": 555, "test": 430},
        "metric": "Spearman correlation", "display_scale": 1.0,
        "score_label": "Test Spearman correlation", "difference_label": "Difference in test Spearman",
    },
    "ecoli_proteins": {
        "task": "classification", "n_outputs": 3, "label": "E. coli expression classes",
        "splits": {"train": 3010, "val": 728, "test": 712},
        "metric": "accuracy", "display_scale": 100.0,
        "score_label": "Test accuracy (%)", "difference_label": "Accuracy difference (percentage points)",
    },
}
# Bind the selected benchmark once for all later checks and display labels.
benchmark_settings = BENCHMARK_SETTINGS[BENCHMARK]
TASK = benchmark_settings["task"]
EXPECTED_OUTPUTS = benchmark_settings["n_outputs"]
EXPECTED_SPLITS = benchmark_settings["splits"]
METRIC = benchmark_settings["metric"]
DISPLAY_SCALE = benchmark_settings["display_scale"]
SCORE_MINIMUM = 0.0 if TASK == "classification" else -1.0
# A cohort here is a historical run group with matching recorded identities.
COHORT_FIELDS = (
    "version", "model_kwargs", "widths", "n_outputs", "inputs", "environment", "code_sha256",
)
# Columns in each cached token vector, separate from the target/output count.
EXPECTED_WIDTHS = {"dna": 512, "rna": 640, "prot": 320}


### Identify the recorded experiment

All candidates use the same retained examples, supplied training/validation/test partitions and frozen NT-v2 100M, RNA-FM and ESM-2 8M embeddings. In the producing runs, the fusion components and TextCNN prediction heads were trained. A *training seed* identifies one realization of that training; it does not select another dataset split, and the same seed across different architectures does not imply identical weights or dropout masks.

The recorded recipe follows the project's implementation of [BioLangFusion, Appendix A.4, List 3](../../Papers/core/BioLangFusion.pdf): Adam optimization, mean squared error for fungal regression or cross-entropy for E. coli classification. Validation Spearman correlation or accuracy governed scheduling, early stopping and selection of the saved epoch. The paper does not fully specify the monitored quantity.

This comparison uses appendix transposed convolution and 100 TextCNN channels. The convolution maps DNA token representations toward codon positions before fusion. Equation 1 and the appendix specify different convolution settings, and the paper's MIL listing uses a different channel width. These choices identify which completed runs belong here; matching final array sizes or favorable test scores does not make the alternatives equivalent.

For each training seed, the saved comparison contains DNA, RNA and protein predictors; concatenation; gated MIL at each positive entropy weight; MIL without entropy; and cross-attention. MIL is the gated method that learns weights for the representations. The entropy weight, written as lambda, controls its additional training penalty. The reader later selects one positive-lambda MIL fit and one single-encoder baseline using validation scores.

The eligible single encoders here use their original token resolutions. The [main Phase A notebook](tier2_blf_phaseA.ipynb) additionally compares RNA-FM averaged over consecutive triplets of nucleotide embeddings, giving one representation per codon. Its selection, repetition and seed-averaged uncertainty procedure remain separate from this appendix comparison.


In [ ]:
# These settings identify the recorded appendix comparison, not a new training request.
BASE_CONFIG = train.TrainConfig(
    lr=3e-5, weight_decay=1e-5, batch_size=32, max_epochs=500,
    early_stopping_patience=20, plateau_patience=5,
    channels=100, tconv="appendix", entropy_lambda=0.0, seed=0,
)

# Select a historical reference explicitly; an absent file does not authorize a new fit.
REFERENCE_RESULT_PATH = phasea_runner.result_path(
    RESULT_DIRECTORY, BENCHMARK, "concat", BASE_CONFIG
)


**List the fits required for this comparison.** Each row of `plan` names one candidate at one training seed. The list is the expected historical comparison grid, used later to check file coverage and resolve exact validation ties. It is not a training queue.


In [ ]:
# Only these native-resolution models can be the comparator in this recorded analysis.
BASELINE_VARIANTS = ("unimodal:dna", "unimodal:rna", "unimodal:prot")

# Declared order also breaks exact validation ties: DNA, RNA, protein; then lower lambda.
CANDIDATES = [
    ("DNA", "unimodal:dna", 0.0),
    ("RNA", "unimodal:rna", 0.0),
    ("Protein", "unimodal:prot", 0.0),
    ("Concat", "concat", 0.0),
    *[(f"MIL lambda={weight:g}", "mil", weight) for weight in POSITIVE_LAMBDAS],
    ("MIL without entropy", "mil", 0.0),
    ("Cross", "cross", 0.0),
]
plan = pd.DataFrame(
    [(seed, label, variant, weight) for seed in SEEDS for label, variant, weight in CANDIDATES],
    columns=["seed", "candidate", "variant", "entropy_lambda"],
)
KEY_COLUMNS = ["seed", "variant", "entropy_lambda"]
planned_keys = set(plan[KEY_COLUMNS].itertuples(index=False, name=None))
print(f"Historical comparison: {benchmark_settings['label']}, {BASE_CONFIG.tconv} convolution, "
      f"{len(plan)} expected records across {len(SEEDS)} seeds. Training disabled.")
print("Metric:", METRIC, "| results:", RESULT_DIRECTORY, "| optional headers:", CACHE_ROOT)


---

## Identify the examples behind the saved runs

The [shared dataset loader](../shared_code/datasets.py) trims surrounding whitespace, uppercases sequences and retains rows at most 999 nucleotides long. It preserves the supplied training, validation and test assignments. The next cell reads the selected benchmark CSV and checks its retained split counts and targets against Setup.

Fungal expression uses a numerical target and is scored by Spearman correlation. E. coli uses integer expression-class IDs 0, 1 and 2 and is scored by accuracy. These are prediction targets; the DNA, RNA and protein representations are model inputs. The numeric class IDs alone do not recover the assay thresholds or their low/medium/high ordering. The [benchmark target guide](../../docs/methods/biological-targets-and-controls.md#fungal-and-e-coli-benchmark-targets) distinguishes source-assay evidence from unresolved local label and row correspondence.

Matching row counts is insufficient to identify the examples. A fingerprint is a compact identifier calculated from contents: here it includes every retained sequence, length, target and split in order. The later result checks use this identifier so reordered rows or changed targets cannot pass merely because the table has the expected size.

The [Stage 3 dataset audit](../stage3/01_blf_datasets.ipynb) retains the preparation qualifications: the local fungal file has 7,089 raw rows versus 7,056 reported in the paper, and matching the retained count does not prove matching examples. The local code filters out long sequences, while the paper describes truncation. Those differences limit a claim of exact reproduction.


In [ ]:
# Validate the selected task and supplied split counts before matching results.
spec = datasets.DATASETS[BENCHMARK]
rows = datasets.blf_benchmark_rows(BENCHMARK)
split_counts = rows[spec.split_column].value_counts().to_dict()
if spec.task != TASK or split_counts != EXPECTED_SPLITS:
    raise ValueError(f"Review the benchmark task or split changes before comparing: {split_counts}")
labels = rows[spec.label_column].to_numpy()
if not np.issubdtype(labels.dtype, np.number) or not np.isfinite(labels).all():
    raise ValueError("The retained target column must contain finite numeric values.")
if TASK == "classification":
    if not np.equal(labels, np.floor(labels)).all() or set(labels) != {0, 1, 2}:
        raise ValueError("E. coli requires the integer class IDs 0, 1, and 2.")
    display(pd.crosstab(rows[spec.split_column], rows[spec.label_column]).reindex(EXPECTED_SPLITS))

# Fingerprint row contents and order together, not just the number of examples.
row_columns = ["sequence", "length", spec.label_column, spec.split_column]
row_texts = (
    json.dumps(row, ensure_ascii=False, separators=(",", ":"), default=lambda value: value.item())
    for row in rows[row_columns].itertuples(index=False, name=None)
)
rows_sha256 = input_fingerprint(row_texts)
# GridDataset preserves test-row order and casts regression targets to float32, classes to int64.
test_rows = rows.loc[rows[spec.split_column] == "test"]
expected_test_truth = test_rows[spec.label_column].to_numpy(
    dtype=np.int64 if TASK == "classification" else np.float32
)
display(pd.Series(split_counts, name="retained rows").reindex(EXPECTED_SPLITS).to_frame())


### Connect the examples to the encoder inputs

The row checks identify the benchmark examples. Cache headers connect those examples to the encoder inputs used to produce embeddings. Each `meta.json` header records an encoder, a sequence count and a fingerprint of the ordered inputs after encoder-specific preparation.

The next cell checks the headers under `CACHE_ROOT` against the selected benchmark. A verified header can be compared with the header hash stored in each result. Missing or invalid headers remain visible in the table and prevent the final comparison from proceeding.

This is a check of metadata files. It does not freshly hash the embedding shards or recover missing historical encoder provenance. Likewise, the code and environment that read a result today may differ from those that produced it. The result records and [execution history](../../docs/run-records.md) retain that distinction.


In [ ]:
# Missing cache headers are a pending verification, not permission to invent their contents.
cache_headers = {}
cache_header_checks = []
for modality, encoder_name in blf.ENCODER_OF_MODALITY.items():
    # Read small metadata only; embedding shards stay unopened.
    metadata_path = CACHE_ROOT / BENCHMARK / encoder_name / "meta.json"
    if not metadata_path.is_file():
        cache_header_checks.append({
            "modality": modality, "status": "pending", "detail": "Cache header is unavailable.",
        })
        continue
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        expected_input = input_fingerprint(
            inputs_for_rows(ENCODERS[encoder_name], rows["sequence"])
        )
        if (metadata.get("encoder") != encoder_name
                or metadata.get("n_sequences") != len(rows)
                or metadata.get("input_sha256") != expected_input):
            raise ValueError("Header does not identify the current encoder inputs.")
        shard_count = metadata.get("n_shards")
        if type(shard_count) is not int or shard_count <= 0:
            raise ValueError("Expected a positive shard count.")
        cache_headers[modality] = {
            "meta_sha256": file_fingerprint(metadata_path),
            "filenames": {"meta.json", *(f"shard_{index:05d}.npz" for index in range(shard_count))},
        }
        cache_header_checks.append({
            "modality": modality, "status": "verified", "detail": "Header matches ordered encoder inputs.",
        })
    except (OSError, ValueError, TypeError, AttributeError) as error:
        cache_header_checks.append({"modality": modality, "status": "invalid", "detail": str(error)})

CACHE_HEADERS_VERIFIED = (
    set(cache_headers) == set(EXPECTED_WIDTHS)
    and all(check["status"] == "verified" for check in cache_header_checks)
)
display(pd.DataFrame(cache_header_checks).set_index("modality"))
print("Cache headers checked." if CACHE_HEADERS_VERIFIED
      else "Independent cache-header verification is pending or failed; final comparison stays blocked.")

# Require the producer's complete source list, without replacing it with this reader's hashes.
source_names = (
    "train.py", "blf.py", "codon_grid.py", "heads.py", "textcnn.py",
    "token_embeddings.py", "../datasets.py", "../sequences.py", "../encoders.py",
)


---

## Identify comparable completed runs

The inputs are now defined. Next, check which final result JSON files describe this benchmark and the recorded appendix experiment. Checkpoints and `.partial` files are not treated as completed results. A missing or excluded file describes the local evidence available to this reader; it does not establish that a fit failed or needs repeating.

Three checks serve different purposes: the recorded configuration and input identity establish which experiment was run; the score history checks the selected epoch; and saved predictions support comparison on individual test rows. A result must also agree with the explicitly selected reference on its recorded source, inputs and producing environment. This keeps the historical run group together.

### Check the recorded settings and input identity

The validator below matches the variant, seed, entropy weight, full training configuration, encoder widths, output size and ordered benchmark rows. It also requires complete recorded source, cache and environment information. Equation 1 records and older records without this identity remain outside the declared appendix comparison, even if their files sit in the inspected directory.


In [ ]:
def check_protocol(result):
    """Require the declared experiment and complete recorded identity; return its grid key."""
    # Match this fit to a declared candidate and the complete training settings.
    config = result.config
    key = (config["seed"], result.variant, config["entropy_lambda"])
    if type(config["seed"]) is not int or key not in planned_keys:
        raise ValueError("Variant, seed, or entropy weight is outside the planned grid.")
    expected_config = asdict(replace(BASE_CONFIG, seed=key[0], entropy_lambda=key[2]))
    if result.benchmark != BENCHMARK or config != expected_config:
        raise ValueError("Benchmark or full training configuration differs from the protocol.")

    # Require a supported identity and the same model dimensions.
    identity = result.identity
    if (not isinstance(identity, dict)
            or type(identity.get("version")) is not int
            or identity["version"] != 1):
        raise ValueError("Missing or unsupported run identity; legacy results are excluded.")
    json.dumps(identity, allow_nan=False)
    if identity.get("variant") != result.variant or identity.get("config") != config:
        raise ValueError("Result and embedded identity disagree.")
    if (identity.get("widths") != EXPECTED_WIDTHS or identity.get("n_outputs") != EXPECTED_OUTPUTS
            or identity.get("model_kwargs") != {}):
        raise ValueError("Model dimensions or arguments differ.")

    # These hashes describe the historical producer, not the currently imported reader.
    code_hashes = identity.get("code_sha256")
    if not isinstance(code_hashes, dict) or set(code_hashes) != set(source_names):
        raise ValueError("The recorded producer source list is incomplete or unexpected.")

    # Sequences, targets and split labels must match in their recorded order.
    inputs = identity["inputs"]
    expected_inputs = {
        "benchmark": BENCHMARK, "task": TASK, "row_columns": row_columns,
        "rows_sha256": rows_sha256, "input_identity_status": "recorded_inputs",
    }
    if any(inputs.get(name) != value for name, value in expected_inputs.items()):
        raise ValueError("Ordered rows or input provenance differ from the current benchmark.")

    # Compare recorded cache membership and any locally verified metadata headers.
    cache_files = inputs["cache_files"]
    if not isinstance(cache_files, dict) or set(cache_files) != set(EXPECTED_WIDTHS):
        raise ValueError("The recorded cache modalities are incomplete or unexpected.")
    fingerprint_groups = {"source": code_hashes}
    for modality, hashes in cache_files.items():
        if not isinstance(hashes, dict) or len(hashes) < 2:
            raise ValueError(f"{modality}: missing recorded cache fingerprints.")
        expected_files = {"meta.json", *(f"shard_{index:05d}.npz" for index in range(len(hashes) - 1))}
        if set(hashes) != expected_files:
            raise ValueError(f"{modality}: cache filenames must include metadata and contiguous shards.")
        if modality in cache_headers:
            header = cache_headers[modality]
            if set(hashes) != header["filenames"] or hashes["meta.json"] != header["meta_sha256"]:
                raise ValueError(f"{modality}: recorded cache files or header hash differ.")
        fingerprint_groups[modality] = hashes
    # Check fingerprint format; this does not reopen or rehash the large shards.
    for name, hashes in fingerprint_groups.items():
        if any(not isinstance(value, str) or len(value) != 64
               or any(character not in "0123456789abcdef" for character in value)
               for value in hashes.values()):
            raise ValueError(f"{name}: missing or malformed SHA256 fingerprints.")

    # Validate the producing environment record, not the current reader machine.
    environment = identity.get("environment")
    required_environment = {
        "python", "platform", "torch", "numpy", "scipy", "device", "threads", "interop_threads",
        "float32_matmul_precision", "deterministic_algorithms", "cudnn_deterministic",
        "cudnn_benchmark", "cudnn_allow_tf32", "cuda_matmul_allow_tf32", "cublas_workspace_config",
    }
    if isinstance(environment, dict) and str(environment.get("device")).startswith("cuda"):
        required_environment.update(("cuda", "device_name", "device_index", "cuda_device_count"))
    if not isinstance(environment, dict) or not required_environment <= environment.keys():
        raise ValueError("The producing environment record is incomplete.")
    if any(not isinstance(environment[name], str) or not environment[name]
           for name in ("python", "platform", "torch", "numpy", "scipy", "device")):
        raise ValueError("The producing environment has invalid version or device fields.")
    if any(type(environment[name]) is not int or environment[name] <= 0
           for name in ("threads", "interop_threads")):
        raise ValueError("The producing environment has invalid thread counts.")
    return key


**Keep one historical run group together.** Individual records can pass those checks while coming from different producing code or environments. The next helper compares each candidate with the reference result using `COHORT_FIELDS`. It checks shared recorded identities while allowing the candidate's seed, fusion method and entropy weight to vary as declared in `plan`.


In [ ]:
def check_reference_cohort(result, reference_identity):
    """Keep the recorded producer, inputs, and environment together across candidates."""
    for field in COHORT_FIELDS:
        if result.identity[field] != reference_identity[field]:
            raise ValueError(f"Recorded {field} differs from the explicitly selected reference cohort.")


### Check the scores and selected training epoch

`Result.load()` reads a saved record; the next validator checks whether its scores and training history agree. Validation and test scores must be finite and lie between -1 and 1 for Spearman correlation, or between 0 and 1 for accuracy.

The saved epoch must be the first epoch with the highest defined validation score. The history must contain consecutive epochs, and a run with no defined validation score cannot supply a selected model. Epoch indices in the saved record start at zero; the displayed table adds one so that epoch 1 means the first training epoch.

Accuracy remains a proportion during checking and model selection. Display copies convert accuracy to percentages and score differences to percentage points.


In [ ]:
def check_scores(result):
    """Reject incomplete or inconsistent scores before any validation selection."""
    for score in (result.val_score, result.test_score):
        if type(score) not in (int, float) or not np.isfinite(score) or not SCORE_MINIMUM <= score <= 1:
            raise ValueError(
                "Validation and test scores must be finite and inside the declared metric range."
            )
    if type(result.epochs) is not int or not 1 <= result.epochs <= BASE_CONFIG.max_epochs:
        raise ValueError("Invalid completed epoch count.")
    if (len(result.history) != result.epochs
            or [entry["epoch"] for entry in result.history] != list(range(result.epochs))
            or type(result.best_epoch) is not int or not 0 <= result.best_epoch < result.epochs):
        raise ValueError("Epoch count, history, or selected epoch is inconsistent.")
    # Reconstruct epoch selection from validation scores, without using test scores.
    validation_history = [entry["val_score"] for entry in result.history]
    for score in validation_history:
        if score is not None and (type(score) not in (int, float)
                                  or not np.isfinite(score) or not SCORE_MINIMUM <= score <= 1):
            raise ValueError("Invalid validation score in the epoch history.")
    # Undefined early scores cannot win; argmax keeps the first exact maximum.
    comparable = [-np.inf if score is None else score for score in validation_history]
    if (result.best_epoch != int(np.argmax(comparable))
            or result.val_score != comparable[result.best_epoch]):
        raise ValueError("Saved validation selection disagrees with the first best epoch.")


### Check predictions against the test examples

A saved score can describe a run without retaining its individual predictions. Paired uncertainty needs more: one usable target and prediction for every test row, in the expected order. The [runner's shared validation helper](run_blf_phaseA.py) checks lengths, finite values, classification IDs and agreement between the predictions' recomputed score and the stored score.

The expected targets use the same numerical representation as the producing loader: float32 for regression and int64 for classification. Score agreement is checked within `1e-12`. Equal labels alone do not prove row identity, because many examples can share a class; the ordered-input fingerprint supplies the accompanying row check.

A record with missing predictions may still supply aggregate-score evidence. A record with present but contradictory predictions is excluded from comparison. The inventory preserves both cases so missing evidence is distinguishable from inconsistent evidence.


In [ ]:
def check_predictions(result):
    """Check stored arrays against the identified test rows, using the runner's same rule."""
    phasea_runner.validate_prediction_arrays(result, TASK, expected_test_truth)

### Read the reference and candidate files

The validators above define the checks. The next cells apply them: first establish the reference result, then inspect each final JSON file, then present compatible and excluded records. Reading a historical result does not require today's reader source to match the old producer's source.


In [ ]:
# Snapshot final files once; another process may publish additional results after this cell.
result_paths = sorted(RESULT_DIRECTORY.glob("*.json"))
inspected = []
# Keep individually checked records visible even if reference matching excludes them.
checked = []
excluded = []
results_by_path = {}
reference_identity = None
try:
    reference_result = train.Result.load(REFERENCE_RESULT_PATH)
    check_protocol(reference_result)
    check_scores(reference_result)
    reference_identity = reference_result.identity
except (OSError, ValueError, TypeError, KeyError, AttributeError) as error:
    print(f"Reference is pending or invalid: {REFERENCE_RESULT_PATH.name}: {error}")


**Inspect each candidate.** Retain individually checked records for the inventory, then separately determine whether they can join the reference run group. Hashing each result before and after reading catches a file that changes during inspection.


In [ ]:
for path in result_paths:
    try:
        # Hash on both sides of reading to detect a file that changes mid-inspection.
        digest_before = file_fingerprint(path)
        result = train.Result.load(path)
        seed, variant, entropy_weight = check_protocol(result)
        check_scores(result)
        prediction_status, prediction_detail = "verified", "Ordered arrays and saved score agree."
        try:
            check_predictions(result)
        except (ValueError, TypeError, KeyError, AttributeError) as error:
            prediction_status = ("missing" if result.test_truth is None or result.test_prediction is None
                                 else "invalid")
            prediction_detail = str(error)
        if file_fingerprint(path) != digest_before:
            raise ValueError("The result changed while being read; refresh the snapshot.")
        # Display epochs from 1; preserve the saved zero-based epoch in the result.
        record = {
            "seed": seed, "variant": variant, "entropy_lambda": entropy_weight,
            "val_score": result.val_score, "test_score": result.test_score,
            "best_epoch": result.best_epoch + 1, "path": path.relative_to(REPO_ROOT).as_posix(),
            "identity": result.identity, "file_sha256": digest_before,
            "has_test_truth": result.test_truth is not None,
            "has_test_prediction": result.test_prediction is not None,
            "prediction_status": prediction_status, "prediction_detail": prediction_detail,
        }
        # Missing predictions remain aggregate evidence; invalid arrays are excluded.
        checked.append(record)
        if prediction_status == "invalid":
            raise ValueError(f"Invalid saved prediction arrays: {prediction_detail}")
        if reference_identity is None:
            raise ValueError(
                "The explicit reference is unavailable or invalid; cohort verification is blocked."
            )
        check_reference_cohort(result, reference_identity)
        inspected.append(record)
        results_by_path[record["path"]] = result
    except (OSError, ValueError, TypeError, KeyError, AttributeError) as error:
        excluded.append({"file": path.name, "reason": str(error)})


**Read the file inventory.** Exclusion reasons explain which records cannot enter this comparison. The prediction table distinguishes fields being present from their arrays passing validation. This determines which results can supply scores alone and which can also support the paired uncertainty calculation.


In [ ]:
result_columns = [
    *KEY_COLUMNS, "val_score", "test_score", "best_epoch", "path", "identity", "file_sha256",
    "has_test_truth", "has_test_prediction", "prediction_status", "prediction_detail",
]
# checked_results passed individual protocol/history checks; results also match the reference cohort.
checked_results = pd.DataFrame(checked, columns=result_columns)
results = pd.DataFrame(inspected, columns=result_columns)
print(
    f"{len(result_paths)} completed files inspected; "
    f"{len(results)} match the protocol and recorded reference cohort."
)
print("Reference:", REFERENCE_RESULT_PATH.relative_to(REPO_ROOT).as_posix())
if excluded:
    display(pd.DataFrame(excluded))

# Presence remains separate from the stricter array validation reported beside it.
prediction_availability = checked_results.groupby(
    ["has_test_truth", "has_test_prediction"]
).size().rename("matching_records").reset_index()
print("Saved per-row fields among protocol/history-checked records (presence only):")
display(prediction_availability.rename(columns={
    "has_test_truth": "test targets stored", "has_test_prediction": "test predictions stored",
    "matching_records": "records",
}))
if not checked_results.empty:
    display(checked_results[[*KEY_COLUMNS, "prediction_status", "prediction_detail"]])


### Check whether the full comparison is represented

Validation selection requires the complete declared candidate set. Otherwise a missing MIL setting or single-encoder fit could change which model appears best. The coverage table therefore checks each candidate and training seed for exactly one compatible result.

| Coverage status | Meaning |
| --- | --- |
| `represented` | Exactly one record passes the result checks and matches the reference run group. |
| `not_represented` | No compatible record qualifies; a file may still be present but excluded. |
| `duplicate` | More than one compatible record qualifies, so the reader cannot choose one unambiguously. |

`GRID_COMPLETE` requires every declared fit exactly once within one historical run group. `COMPARISON_READY` additionally requires verified cache headers for every input representation. These are prerequisites for this reader's selection and displays, not measures of overall team progress. File recency or a higher test score cannot resolve duplicate evidence.


In [ ]:
# Match each planned candidate/seed to the number of eligible result files.
counts = results.groupby(KEY_COLUMNS).size().rename("matching_files").reset_index()
coverage = plan.merge(counts, on=KEY_COLUMNS, how="left")
coverage["matching_files"] = coverage["matching_files"].fillna(0).astype(int)
coverage["status"] = coverage["matching_files"].map(
    lambda count: "not_represented" if count == 0 else "represented" if count == 1 else "duplicate"
)
display(coverage.pivot(index="candidate", columns="seed", values="status").reindex(
    [label for label, _, _ in CANDIDATES]
))

# Compare recorded identities, not this reader's code or current kernel environment.
cohort_signatures = results["identity"].map(
    lambda identity: json.dumps(
        {field: identity[field] for field in COHORT_FIELDS},
        sort_keys=True, allow_nan=False,
    )
)
one_cohort = cohort_signatures.nunique() == 1
if cohort_signatures.nunique() > 1:
    print("Comparison blocked: more than one recorded input/code/environment cohort.")

# Complete result coverage and independent header verification are separate gates.
GRID_COMPLETE = coverage["matching_files"].eq(1).all() and one_cohort
COMPARISON_READY = GRID_COMPLETE and CACHE_HEADERS_VERIFIED
print(
    f"{coverage['matching_files'].eq(1).sum()}/{len(plan)} "
    "planned candidates are represented exactly once."
)
print("Independent cache-header verification:", "verified" if CACHE_HEADERS_VERIFIED else "pending or failed")
print("Ready for the declared comparison." if COMPARISON_READY
      else "Comparison pending: resolve missing, duplicate, incompatible, or unverified inputs.")


### Locate saved work when coverage is incomplete

The coverage table describes eligibility for comparison. The inventory below explains what was found at the expected locations: a completed record, missing or invalid predictions, an unavailable reference, an incompatible result, a checkpoint, or no local file.

For example, `reference_pending` means a saved fit exists but its reference run group cannot yet be established. `not_found_locally` means no result or checkpoint was found at the inspected path; work may exist elsewhere. These statuses help locate the evidence to review without replacing it.

This inspection reads records and checks file presence. It does not load checkpoints, open token arrays or test whether a historical fit can resume under today's code. The retained runner's execution checks are a separate operation.


In [ ]:
# Inspect local evidence only; retain the table name for existing consumers.
preview_rows = []
for item in plan.itertuples(index=False):
    config = replace(BASE_CONFIG, seed=item.seed, entropy_lambda=item.entropy_lambda)
    expected_path = phasea_runner.result_path(RESULT_DIRECTORY, BENCHMARK, item.variant, config)
    key_mask = (checked_results["seed"].eq(item.seed)
                & checked_results["variant"].eq(item.variant)
                & checked_results["entropy_lambda"].eq(item.entropy_lambda))
    stored = checked_results.loc[key_mask]
    matched = results.loc[
        results["seed"].eq(item.seed) & results["variant"].eq(item.variant)
        & results["entropy_lambda"].eq(item.entropy_lambda)
    ]
    # Distinguish a missing artifact from a present record excluded by the checks.
    if len(stored) > 1:
        status, detail = (
            "duplicate_requires_review",
            "More than one protocol-compatible record.",
        )
    elif len(stored) == 1 and stored.iloc[0]["prediction_status"] != "verified":
        status = ("predictions_absent" if stored.iloc[0]["prediction_status"] == "missing"
                  else "predictions_invalid")
        detail = stored.iloc[0]["prediction_detail"]
    elif len(stored) == 1 and reference_identity is None:
        status, detail = (
            "reference_pending",
            "A saved fit exists; the declared reference must be reviewed.",
        )
    elif len(matched) == 1:
        status, detail = (
            "recorded_evidence_available",
            "Analysis evidence; execution reuse needs the runner's identity check.",
        )
    elif expected_path in result_paths or expected_path.is_file() or len(stored):
        status, detail = (
            "incompatible_with_comparison",
            "A completed record is preserved; review why it differs.",
        )
    elif expected_path.with_suffix(".checkpoint").is_file():
        status, detail = (
            "checkpoint_requires_review",
            "The runner must verify resumability; this reader does not load checkpoints.",
        )
    else:
        status, detail = (
            "not_found_locally",
            "No result or checkpoint found here; work elsewhere may already exist.",
        )
    preview_rows.append({
        "seed": item.seed, "candidate": item.candidate, "variant": item.variant,
        "entropy_lambda": item.entropy_lambda, "status": status, "detail": detail,
        "expected_path": expected_path.relative_to(REPO_ROOT).as_posix(),
    })
execution_preview = pd.DataFrame(preview_rows)
display(execution_preview[["seed", "candidate", "status", "detail"]])
print("Saved-work inspection only; these statuses do not schedule training or establish work elsewhere.")


---

## Select on validation, then compare test scores

When the recorded comparison is ready, choose two things separately within each training seed: the single encoder with the highest validation score, and the positive MIL entropy weight with the highest validation score. Test scores do not make either choice. Exact ties follow the candidate order in Setup. MIL with zero entropy stays a separate ablation, meaning the version with that penalty removed.

**Select the single-encoder baseline.** The eligible baseline is chosen from native DNA, RNA and protein predictors:

$$
b_s = \underset{b\in\{\mathrm{DNA,RNA,Protein}\}}{\operatorname{argmax}}\;S^{\mathrm{val}}_{b,s}.
$$

**In this notation:**

- $s$ identifies the training seed.
- $b$ identifies a candidate single-encoder predictor.
- $S^{\mathrm{val}}_{b,s}$ is that predictor's validation score for seed $s$.
- $\operatorname{argmax}$ returns the candidate with the highest score; $b_s$ is the selected baseline.

The baseline can differ between seeds. This comparison evaluates the repeated selection procedure, rather than assuming one permanently fixed single encoder.

**Compare test performance.** After validation selection, subtract that seed's selected baseline score from the fusion method's score on the same test rows:

$$
\Delta_{m,s} = S^{\mathrm{test}}_{m,s} - S^{\mathrm{test}}_{b_s,s}.
$$

**In this notation:**

- $m$ identifies a reported fusion method.
- $S^{\mathrm{test}}_{m,s}$ is its test score for seed $s$.
- $S^{\mathrm{test}}_{b_s,s}$ is the test score of the baseline already selected on validation.
- $\Delta_{m,s}$ is the fusion-minus-baseline difference for that seed.

A positive difference means the fusion method outscored the validation-selected single encoder. $S$ is Spearman correlation for fungal expression, comparing predicted and target rankings, or accuracy for E. coli, the proportion of correctly predicted classes. Accuracy differences are multiplied by 100 for display as percentage points, not relative percentage improvement.

The producing loop already stored test scores for all candidates. The selection below uses only validation scores, although a supplied benchmark split examined during development still differs from an untouched confirmatory evaluation.


In [ ]:
# Reset derived tables on every run so an incomplete refresh cannot retain an old comparison.
selected = pd.DataFrame()
choices = pd.DataFrame()
paired = pd.DataFrame()
MAIN_METHODS = ["DNA", "RNA", "Protein", "Concat", "MIL", "Cross"]
if COMPARISON_READY:
    ordered_results = plan.merge(results, on=KEY_COLUMNS, how="left", validate="one_to_one")
    selected_rows, choice_rows, difference_rows = [], [], []
    for seed in SEEDS:
        seed_results = ordered_results[ordered_results["seed"] == seed]
        single_runs = seed_results[seed_results["variant"].isin(BASELINE_VARIANTS)]
        positive_mil = seed_results[
            seed_results["variant"].eq("mil") & seed_results["entropy_lambda"].isin(POSITIVE_LAMBDAS)
        ]
        # Plan order breaks exact ties; neither choice uses the test score.
        baseline = single_runs.loc[single_runs["val_score"].idxmax()]
        chosen_mil = positive_mil.loc[positive_mil["val_score"].idxmax()]
        choice_rows.append({
            "seed": seed, "single encoder": baseline["candidate"],
            "single validation": baseline["val_score"], "MIL lambda": chosen_mil["entropy_lambda"],
            "MIL validation": chosen_mil["val_score"],
        })
        # Fixed candidates plus the one validation-selected positive-lambda MIL run.
        reported_runs = pd.concat([
            seed_results[seed_results["entropy_lambda"].eq(0)],
            chosen_mil.to_frame().T,
        ])
        for _, run in reported_runs.iterrows():
            method = "MIL" if run["variant"] == "mil" and run["entropy_lambda"] > 0 else run["candidate"]
            selected_rows.append({
                "seed": seed, "method": method,
                "val_score": run["val_score"], "test_score": run["test_score"],
                "best_epoch": run["best_epoch"], "path": run["path"],
            })
            if method in ("Concat", "MIL", "Cross", "MIL without entropy"):
                difference_rows.append({"seed": seed, "method": method,
                                        "delta": run["test_score"] - baseline["test_score"]})
    selected = pd.DataFrame(selected_rows)
    choices = pd.DataFrame(choice_rows)
    paired = pd.DataFrame(difference_rows)
    # Convert a display copy; keep stored and calculation scores in their native units.
    choice_display = choices.set_index("seed").copy()
    choice_display[["single validation", "MIL validation"]] *= DISPLAY_SCALE
    print("Validation-selection display units:", benchmark_settings["score_label"].replace("Test ", ""))
    display(choice_display.round(4))
else:
    print("No model selection: the full planned grid is not ready.")


### Read individual scores before summarizing seeds

First read the selection table above: it identifies the baseline and positive MIL weight selected independently for each training seed. The next tables retain each model's seed, validation score, test score and selected epoch, then summarize the test scores across seeds.

The summary gives an arithmetic mean and sample standard deviation (`ddof=1`). The standard deviation describes variation among training realizations on this fixed dataset split, including model selection where it is part of the procedure. It is not a confidence interval or variation among biological replicates.

Spearman scores retain their correlation scale. Accuracy is displayed as a percentage; its standard deviation and score differences are in percentage points. The last table compares MIL without entropy with the same validation-selected baseline. A better ablation test score does not replace the MIL model selected by the positive-lambda rule.


In [ ]:
if COMPARISON_READY:
    # Scale copied tables only; original results retain native score units.
    # Accuracy remains a proportion; fungal expression uses a correlation.
    score_display = selected.drop(columns="path").sort_values(["method", "seed"]).copy()
    score_display[["val_score", "test_score"]] *= DISPLAY_SCALE
    print("Score display units:", benchmark_settings["score_label"].replace("Test ", ""))
    display(score_display.round(4))
    # Summarize training-seed variation, not independent biological replicates.
    summary = selected.groupby("method")["test_score"].agg(["count", "mean", "std"])
    summary_display = (
        summary.reindex(MAIN_METHODS)
        .rename(columns={"count": "seeds", "std": "seed SD"})
        .copy()
    )
    summary_display[["mean", "seed SD"]] *= DISPLAY_SCALE
    display(summary_display.round(4))
    # Keep the zero-entropy comparison separate from validation-selected MIL.
    ablation = selected[selected["method"] == "MIL without entropy"].merge(
        paired, on=["seed", "method"], validate="one_to_one"
    )
    ablation_display = ablation[["seed", "val_score", "test_score", "delta"]].copy()
    ablation_display[["val_score", "test_score", "delta"]] *= DISPLAY_SCALE
    display(ablation_display.round(4))
else:
    print("Score summaries wait for a complete, comparable grid.")


### Compare the selected methods visually

The score tables show how performance varies between methods and between training seeds. The first figure keeps each seed visible and marks the mean. The second compares each fusion score with that seed's validation-selected single-encoder score; its zero line means equal performance. Seed colors stay consistent across the figures.

These figures use aggregate scores. Their points can be compared descriptively with [BioLangFusion's Table 1](../../Papers/core/BioLangFusion.pdf), while retaining the dataset and implementation qualifications. Closeness to a published value does not establish exact reproduction.

A mean gain and its consistency across seeds do not yet describe uncertainty from the finite test sample. The later bootstrap uses saved per-row predictions to examine that separate question.


In [ ]:
if COMPARISON_READY:
    score_matrix = selected.pivot(index="method", columns="seed", values="test_score").reindex(MAIN_METHODS)
    score_matrix = score_matrix * DISPLAY_SCALE
    positions = np.arange(len(MAIN_METHODS))
    offsets = np.linspace(-0.15, 0.15, len(SEEDS))
    seed_colors = ("#0072B2", "#D55E00", "#009E73")
    figure, axis = plt.subplots(figsize=(9, 4.5), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        axis.scatter(positions + offset, score_matrix[seed], label=f"Seed {seed}", color=color, s=48)
    axis.scatter(
        positions, score_matrix.mean(axis=1),
        marker="_", s=250, color="black", label="Mean", zorder=3,
    )
    axis.set(xticks=positions, xticklabels=MAIN_METHODS, ylabel=benchmark_settings["score_label"],
             title=f"{benchmark_settings['label']}: {len(SEEDS)} training seeds on the supplied split")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.show()
else:
    print("Performance figure pending the complete grid.")

**Read the paired gains.** Each point below subtracts the single-encoder baseline selected for that same seed. The comparison therefore follows the validation-selection procedure even when the selected baseline changes between seeds. A point above zero favors fusion; the bootstrap section examines uncertainty around that difference.


In [ ]:
if COMPARISON_READY:
    fusion_methods = ["Concat", "MIL", "Cross"]
    delta_matrix = paired.pivot(index="method", columns="seed", values="delta").reindex(fusion_methods)
    delta_matrix = delta_matrix * DISPLAY_SCALE
    positions = np.arange(len(fusion_methods))
    figure, axis = plt.subplots(figsize=(8, 4.5), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        axis.scatter(positions + offset, delta_matrix[seed], label=f"Seed {seed}", color=color, s=48)
    axis.scatter(
        positions, delta_matrix.mean(axis=1),
        marker="_", s=250, color="black", label="Mean", zorder=3,
    )
    axis.axhline(0, color="gray", linestyle="--", linewidth=1)
    axis.set(xticks=positions, xticklabels=fusion_methods, ylabel=benchmark_settings["difference_label"],
             title="Fusion minus the validation-selected single encoder, within each seed")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.show()
else:
    print("Paired-difference figure pending the complete grid.")

### Assess uncertainty from the finite test sample

For each training seed, compare a selected fusion model with that seed's validation-selected baseline. Draw test-row positions with replacement, use the same positions for both models, and calculate their score difference. Repeat this for `BOOTSTRAP_DRAWS` resamples; the 2.5th and 97.5th percentiles give the displayed interval. `BOOTSTRAP_SEED` makes the resampling repeatable without training another model.

These are **per-training-seed intervals for fixed fitted models on this test split**. They do not include retraining variability, a new validation selection, a different population or dependence among biologically related sequences. The earlier standard deviation across training seeds describes a different source of variation. The same test rows are never stacked across seeds as if they were additional independent observations.

Each pair needs verified saved predictions and a ready comparison. Missing arrays or an undefined resampled correlation produce an unavailable result. The intervals are individual descriptive intervals without multiple-comparison correction. The fraction of resampled differences above zero is a resampling summary, not a posterior probability that a biological hypothesis is true.


In [ ]:
uncertainty_rows = []
if COMPARISON_READY:
    for seed in SEEDS:
        seed_results = selected[selected["seed"] == seed].set_index("method")
        # Reuse the baseline already chosen on validation for this training seed.
        baseline_name = choices.loc[choices["seed"] == seed, "single encoder"].iloc[0]
        baseline_path = seed_results.loc[baseline_name, "path"]
        baseline_result = results_by_path[baseline_path]
        for method in paired.loc[paired["seed"] == seed, "method"]:
            fusion_path = seed_results.loc[method, "path"]
            fusion_result = results_by_path[fusion_path]
            row = {"seed": seed, "method": method, "baseline": baseline_name,
                   "fusion_path": fusion_path, "baseline_path": baseline_path,
                   "status": "unavailable", "detail": ""}
            try:
                check_predictions(fusion_result)
                check_predictions(baseline_result)
                # Resample the same test positions for both models; no fit occurs.
                interval = train.paired_bootstrap(
                    fusion_result, baseline_result, TASK,
                    draws=BOOTSTRAP_DRAWS, seed=BOOTSTRAP_SEED,
                )
                row.update(interval, status="available")
            except (ValueError, TypeError, KeyError, AttributeError) as error:
                row["detail"] = str(error)
            uncertainty_rows.append(row)
uncertainty = pd.DataFrame(uncertainty_rows)
UNCERTAINTY_READY = (
    COMPARISON_READY and len(uncertainty) == len(paired) and len(uncertainty) > 0
    and uncertainty["status"].eq("available").all()
)


**Read the interval table.** Each row compares one fusion method and one training seed with that seed's selected baseline. An unavailable row gives its reason. Available intervals describe test-row resampling while the fitted models stay fixed; they do not average away training-seed variation.


In [ ]:
if not uncertainty.empty:
    uncertainty_display = uncertainty.drop(columns=["fusion_path", "baseline_path"]).copy()
    for name in ("difference", "ci_low", "ci_high"):
        if name in uncertainty_display:
            uncertainty_display[name] *= DISPLAY_SCALE
    print(
        benchmark_settings["difference_label"],
        f"| {BOOTSTRAP_DRAWS} draws, bootstrap seed {BOOTSTRAP_SEED}",
    )
    display(uncertainty_display.round(4))
else:
    print("Paired uncertainty waits for a complete, independently checked comparison.")


**Read the intervals beside the observed gains.** The point is the observed fusion-minus-baseline difference; the line marks its individual 95% interval. The zero line means equal performance. An interval crossing zero leaves both directions compatible with this conditional resampling summary. Colors retain the training seeds, and each legend entry names its selected baseline. The zero-entropy MIL ablation remains separately labeled.


In [ ]:
if UNCERTAINTY_READY:
    interval_methods = ["Concat", "MIL", "Cross", "MIL without entropy"]
    positions = np.arange(len(interval_methods))
    interval_figure, interval_axis = plt.subplots(figsize=(9.5, 5.2), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        seed_intervals = (
            uncertainty[uncertainty["seed"] == seed]
            .set_index("method")
            .reindex(interval_methods)
        )
        baseline_name = seed_intervals["baseline"].iloc[0]
        # Draw percentile endpoints directly; the observed difference need not be inside them.
        interval_axis.hlines(
            positions + offset, seed_intervals["ci_low"] * DISPLAY_SCALE,
            seed_intervals["ci_high"] * DISPLAY_SCALE, color=color, linewidth=2,
        )
        interval_axis.scatter(
            seed_intervals["difference"] * DISPLAY_SCALE, positions + offset,
            color=color, s=42, label=f"Seed {seed} vs {baseline_name}", zorder=3,
        )
    interval_axis.axvline(0, color="gray", linestyle="--", linewidth=1)
    interval_axis.set(
        yticks=positions, yticklabels=interval_methods, xlabel=benchmark_settings["difference_label"],
        title=(
            "Fusion minus the validation-selected single encoder\n"
            "Individual conditional 95% paired intervals"
        ),
    )
    interval_axis.invert_yaxis()
    interval_axis.grid(axis="x", alpha=0.25)
    interval_axis.legend(ncol=len(SEEDS), loc="upper center", bbox_to_anchor=(0.5, -0.15))
    plt.show()
else:
    print("Paired-interval figure waits for verified predictions and available intervals for every pair.")

### Retain the exact comparison inputs

The tables and figures depend on which files passed the checks and which candidates validation selected. `analysis_manifest` collects that information: candidate and selected paths and hashes, ordered benchmark identity, recorded settings, validation choices, metric units and bootstrap outcomes. Keeping the losing validation candidates makes the selection traceable as well as the displayed winners.

The record distinguishes the historical producer's identity from the notebook and helpers reading the results now. It also records that this reader did not rehash embedding-shard contents. The manifest is prepared in memory only; this cell does not save or overwrite a file. An exported analysis can retain the reviewed manifest and link it from the existing [run records](../../docs/run-records.md).


In [ ]:
analysis_manifest = None
if COMPARISON_READY:
    result_details = results.set_index("path")
    # Preserve exactly which result files supplied the displayed selections.
    selected_files = [
        {"path": row.path, "sha256": result_details.loc[row.path, "file_sha256"],
         "seed": int(row.seed), "method": row.method,
         "prediction_status": result_details.loc[row.path, "prediction_status"]}
        for row in selected.itertuples(index=False)
    ]
    # Replace unavailable numeric cells with JSON null without rounding available values.
    uncertainty_records = (
        uncertainty.astype(object)
        .where(pd.notna(uncertainty), None)
        .to_dict(orient="records")
    )
    analysis_manifest = {
        "format_version": 1, "kind": "BioLangFusion Phase A analysis of saved results",
        "benchmark": BENCHMARK, "task": TASK, "rows_sha256": rows_sha256,
        "splits": dict(EXPECTED_SPLITS), "metric": METRIC,
        "stored_score_units": "accuracy proportion" if TASK == "classification" else "Spearman correlation",
        "display_scale": DISPLAY_SCALE,
        "protocol": {"base_config": asdict(BASE_CONFIG), "seeds": list(SEEDS),
                     "positive_lambdas": list(POSITIVE_LAMBDAS),
                     "candidates": plan.to_dict(orient="records")},
        "reference_path": REFERENCE_RESULT_PATH.relative_to(REPO_ROOT).as_posix(),
        "recorded_producer_cohort": {key: reference_identity[key] for key in COHORT_FIELDS},
        "cache_headers_verified": bool(CACHE_HEADERS_VERIFIED),
        "cache_shard_bytes_reverified": False,
        # Identify this reader separately from the historical training source.
        "reader_notebook_sha256": file_fingerprint(
            CODE_DIR / "stage4" / "tier2_blf_phaseA_appendix.ipynb"
        ),
        "reader_helper_sha256": {
            "Code/shared_code/fusion/train.py": file_fingerprint(
                CODE_DIR / "shared_code" / "fusion" / "train.py"
            ),
            "Code/stage4/run_blf_phaseA.py": file_fingerprint(CODE_DIR / "stage4" / "run_blf_phaseA.py"),
        },
        "candidate_files": [
            {"path": row.path, "sha256": row.file_sha256, "seed": int(row.seed),
             "variant": row.variant, "entropy_lambda": float(row.entropy_lambda)}
            for row in results.itertuples(index=False)
        ],
        "selected_files": selected_files,
        "validation_choices": choices.to_dict(orient="records"),
        "bootstrap": {"draws": BOOTSTRAP_DRAWS, "seed": BOOTSTRAP_SEED,
                      "method": "paired test-row percentile intervals, separately per training seed",
                      "all_pairs_available": bool(UNCERTAINTY_READY),
                      "comparisons": uncertainty_records},
    }
    # Check that the proposed record can be exported as strict JSON; do not save a file.
    json.dumps(analysis_manifest, allow_nan=False)
    print(
        f"Analysis manifest prepared in memory for {len(selected_files)} selected result files; "
        "no file written."
    )
else:
    print("No analysis manifest: the declared comparison is not ready.")


---

## What this comparison can establish

Read the paired differences to assess whether each fusion method improves on its validation-selected single encoder across the recorded training seeds. Read the bootstrap intervals separately: they describe resampling uncertainty for fixed fitted models on the selected test split. A blocked comparison or unavailable interval supplies no finding about whether fusion helps.

A positive gain supports predictive usefulness for this particular configuration and evaluation. It can reflect complementary predictive information, differences in trainable capacity or optimization. It does not establish biological alignment, a causal mechanism or three independent sources of molecular evidence. The [E. coli execution and analysis record](../../docs/run-records.md#2026-10-19t220959z-phasea-ecoli-three-seed-classification-comparison) retains the findings of that completed historical comparison; they should not be presented as output from a different selected benchmark.

The remaining reproduction limits include exact row correspondence, filtering versus the paper's truncation, local target transformations and class mapping, convolution settings, channel width and the eligible single-encoder baselines. The NT cache width of 512 also remains distinct from the paper's 4,107-feature description. The [dataset audit](../stage3/01_blf_datasets.ipynb) and [target guide](../../docs/methods/biological-targets-and-controls.md#fungal-and-e-coli-benchmark-targets) hold the detailed evidence. An absence of exact sequence overlap does not establish independence from homologous sequences, shared composition or encoder pretraining data.

The [main Phase A notebook](tier2_blf_phaseA.ipynb) already contains the separate Equation 1 and codon-pooled RNA comparison. This appendix preserves how its own completed runs are inspected and compared; it is not a prerequisite for that notebook or the encoder swaps. Differences between the two workflows cannot isolate one architectural effect when configuration, model selection and uncertainty procedures also differ.

When reporting this appendix analysis, retain the selected file paths, validation choices and historical producer information. A source fingerprint identifies recorded bytes but cannot by itself establish which module objects a long-lived kernel executed. The [run records](../../docs/run-records.md) preserve that history, while the [Stage 4 overview](README.md) and [Gantt workbook](../../docs/Research_Project_Plan_Gantt.xlsx), `Details` items 17 and 19, explain the broader research role.
